**ЛАБОРАТОРНАЯ РАБОТА № 4**

**Классификация и оценка классификатора**

# Цель работы

Научиться предсказывать два класса, интерпретировать вероятность класса,
confusion matrix и метрики Accuracy, Precision, Recall и F1.

# Минимум Python для этой работы

**Логическое сравнение** --- даёт True/False; его можно превратить в 1/0.

```python
y = (df['Churn'] == 'Да').astype(int)
```

**Индекс [:, 1]** --- выбирает второй столбец из двумерного массива.

```python
prob = model.predict_proba(X_test)[:, 1]
```

**Условие >= 0.5** --- превращает вероятность в решение 0 или 1.

```python
pred = (prob >= 0.5).astype(int)
```

# 1. Данные и целевая переменная

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("ai_customer_churn_lab4.csv")

display(df.head())
print(df["Churn"].value_counts())

**Задание 1.** Что означает Churn=Да? Почему здесь ответ является классом, а не числом?

**Churn = «Да» означает, что клиент ушёл от компании, то есть перестал пользоваться услугами. Это класс, а не число, потому что ответ --- это выбор одного из двух вариантов: «ушёл» или «остался». Между «Да» и «Нет» нет ничего среднего, клиент не может уйти «наполовину» или «на 0.3». Поэтому это задача классификации, а не регрессии.**

# 2. Преобразование цели

In [ ]:
X = df.drop(columns="Churn")
y = (df["Churn"] == "Да").astype(int)

print(y.value_counts())

> **Почему 0 и 1**  
> Многие алгоритмы удобно обучать на числовой цели: 0 --- клиент остался, 1 --- клиент ушёл.

**Задание 2.** Какой класс встречается реже?

**Реже встречается класс Churn = «Да», то есть ушедшие клиенты. Их гораздо меньше, чем оставшихся (в наших данных ушедших примерно 1 из 10). Значит набор данных несбалансированный, и это надо учитывать при оценке модели.**

# 3. Stratified train/test

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

> **stratify**  
> Сохраняет примерно одинаковую долю классов в train и test.

**Задание 3.** Почему это особенно важно, если один класс встречается заметно реже?

**Параметр stratify сохраняет примерно одинаковую долю классов и в train, и в test. Если один класс редкий, без stratify легко получить ситуацию, когда в тестовую выборку попадёт очень мало таких клиентов или вообще ни одного. Тогда метрики будут случайными и нечестными: модель как будто хорошая, а проверить это не на чем. Со stratify обе выборки по составу похожи на исходные данные.**

# 4. Baseline

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, recall_score

majority = y_train.mode()[0]
baseline_pred = np.full(len(y_test), majority)

print("Accuracy:", accuracy_score(y_test, baseline_pred))
print("Recall:", recall_score(y_test, baseline_pred, zero_division=0))

**Задание 4.** Почему baseline может иметь неплохую Accuracy и одновременно Recall=0 для ухода клиентов?

**Baseline всегда предсказывает самый частый класс --- «не ушёл». Оставшихся клиентов большинство, поэтому модель просто угадывает большинство и Accuracy получается довольно высокой (у нас около 0.89). Но уходящих клиентов она не находит ни одного, поэтому Recall = 0. Получается, что высокая Accuracy здесь обманчива: модель ничего полезного не делает, просто говорит «всё хорошо».**

# 5. Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

# В notebook уже подготовлен Pipeline.
model.fit(X_train, y_train)

pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:, 1]

`predict()` выдаёт класс 0/1. `predict_proba()` выдаёт вероятность.
Например, 0.82 можно читать как высокий риск класса 1.

**Задание 5.** Найдите клиента с вероятностью ухода, ближайшей к 0.5. Почему такой прогноз менее уверенный?

**Клиент с вероятностью около 0.5 --- это пограничный случай: модель почти не отличает его от порога. Если вероятность, скажем, 0.51, клиент попадает в класс «уйдёт», но совсем чуть-чуть. Стоит немного измениться данным --- и он перескочит в другой класс. Чем ближе вероятность к 0.5, тем меньше модель уверена, поэтому таким прогнозам доверяют меньше.**

# 6. Confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, pred)
print(cm)

tn, fp, fn, tp = cm.ravel()
print("TN:", tn, "FP:", fp, "FN:", fn, "TP:", tp)

| Обозначение | Смысл для Churn |
|---|---|
| TP | клиент реально ушёл, модель предупредила |
| TN | клиент остался, модель это определила |
| FP | ложная тревога: модель ожидала уход |
| FN | опасный пропуск: клиент ушёл, но модель не предупредила |

**Задание 6.** Какая ошибка особенно неприятна, если компания хочет заранее удерживать клиентов?

**Неприятнее всего FN --- опасный пропуск. Это когда клиент реально ушёл, а модель сказала, что всё нормально, и компания не успела его удержать. Ошибка FP (ложная тревога) тоже неприятна, но она стоит лишь лишнего звонка или скидки, а FN --- это потерянный клиент. Поэтому для удержания важнее никого не пропустить, пусть даже ценой лишних тревог.**

# 7. Метрики

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score
)

print("Accuracy:", accuracy_score(y_test, pred))
print("Precision:", precision_score(y_test, pred))
print("Recall:", recall_score(y_test, pred))
print("F1:", f1_score(y_test, pred))

| Метрика | Простой вопрос |
|---|---|
| Accuracy | сколько ответов верно вообще? |
| Precision | если модель сказала «уйдёт», как часто это правда? |
| Recall | сколько реально уходящих клиентов мы нашли? |
| F1 | насколько сбалансированы Precision и Recall? |

**Задание 7.** Какую метрику вы бы сделали главной для системы удержания клиентов? Объясните.

**Главной бы сделали Recall (полноту по классу «ушёл»), потому что он показывает, какую долю реально уходящих клиентов мы поймали. Для удержания важнее никого не пропустить. Но смотреть только на Recall нельзя: если гнаться только за ним, будет очень много ложных тревог (страдает Precision). Поэтому на практике я смотрю на Recall, а F1 использую как общую проверку, чтобы Precision и Recall были более-менее сбалансированы.**

# 8. Порог

In [ ]:
for threshold in [0.30, 0.50, 0.70]:
    pred_t = (prob >= threshold).astype(int)
    print(
        threshold,
        precision_score(y_test, pred_t, zero_division=0),
        recall_score(y_test, pred_t, zero_division=0)
    )

**Задание 8.** Что происходит с Recall, если уменьшить порог? Почему число предупреждений обычно растёт?

**Если уменьшить порог (например с 0.5 до 0.3), модель становится подозрительнее: она относит к классу «уйдёт» больше клиентов, даже если вероятность небольшая. Значит предупреждений становится больше, Recall растёт (мы ловим больше уходящих), но Precision обычно падает (среди предупреждений больше ложных). Если поднять порог до 0.7, всё наоборот: предупреждений меньше, Recall падает, Precision растёт.**

# 9. Коэффициенты

В notebook выводится таблица коэффициентов Logistic Regression. Большой по модулю коэффициент означает сильное влияние соответствующего преобразованного признака на решение модели.

**Задание 9.** Назовите три признака с крупными коэффициентами. Почему нельзя по ним автоматически делать вывод о причинности?

**Признаки с крупными коэффициентами у нас --- это, скорее всего, SupportCalls (число обращений в поддержку), MonthlySpend (месячные расходы) и TenureMonths (сколько месяцев клиент с нами). Логично: чем чаще клиент звонит в поддержку и чем больше платит, тем выше риск ухода; а долгий срок обычно снижает риск.**

**Но делать вывод о причинности по коэффициентам нельзя. Коэффициент показывает только связь признака с прогнозом модели, а не то, что признак является причиной ухода. Модель могла просто уловить совпадение, признаки могут быть связаны между собой, а ещё коэффициент относится к уже преобразованному (масштабированному и закодированному) признаку. Чтобы говорить о причине, нужен отдельный эксперимент, а не модель.**

# 10. Итог

**Задание 10.** Напишите 5--7 предложений: лучше ли модель baseline, какие ошибки важнее, какую метрику и какой порог вы выбрали бы.

**Логистическая регрессия оказалась лучше baseline: baseline просто всегда отвечает «не уйдёт», получает приличную Accuracy (около 0.89), но не находит ни одного ушедшего клиента (Recall = 0). Наша модель отвечает верно чуть реже, зато реально находит часть уходящих клиентов, поэтому она полезнее. Самая неприятная ошибка для нас --- FN: когда модель говорит «всё хорошо», а клиент на самом деле уходит, и мы теряем его. Поэтому главной метрикой я выберу Recall (полноту по классу ухода), а F1 буду смотреть, чтобы не было слишком много ложных тревог. Порог я бы взял ниже стандартного 0.5, например 0.3--0.4, чтобы ловить больше уходящих клиентов. При этом надо помнить, что высокая Accuracy здесь ничего не значит из-за несбалансированных классов.**

# Контрольные вопросы

**1. Что такое классификация?**

Классификация --- это задача, где нужно отнести объект к одному из заранее известных классов (например, «уйдёт» или «не уйдёт»). Ответ --- это категория, а не произвольное число, как в регрессии.

**2. Что такое бинарная классификация?**

Бинарная классификация --- это классификация, где классов ровно два (0 и 1, «да» и «нет»). Наша задача как раз такая: ушёл или остался.

**3. Зачем нужен stratify?**

stratify нужен, чтобы доли классов в train и test были такими же, как во всех данных. Особенно это важно при дисбалансе, чтобы редкий класс не потерялся в одной из выборок.

**4. Чем predict_proba() отличается от predict()?**

predict() выдаёт сразу класс (0 или 1). predict_proba() выдаёт вероятности каждого класса; нас интересует вероятность класса 1. Вероятность удобнее, потому что по ней можно самому выбрать порог.

**5. Что такое TP?**

TP (True Positive) --- модель сказала «уйдёт», и клиент действительно ушёл. Это правильное попадание.

**6. Что такое FN?**

FN (False Negative) --- модель сказала «не уйдёт», а клиент на самом деле ушёл. Это пропуск ушедшего клиента.

**7. Чем Precision отличается от Recall?**

Precision --- из всех, кого модель назвала уходящими, сколько реально ушли (качество тревог). Recall --- из всех реально ушедших, сколько мы нашли (полнота поимки). То есть Precision про лишние тревоги, а Recall про пропуски.

**8. Что показывает F1?**

F1 --- это среднее (гармоническое) между Precision и Recall. Он показывает, насколько эти две метрики сбалансированы, и удобен, когда нужно учесть обе сразу.

**9. Зачем менять порог?**

Порог нужен, чтобы превратить вероятность в решение 0 или 1. Меняя порог, мы выбираем, что важнее: ловить больше уходящих клиентов (низкий порог) или меньше ошибаться (высокий порог).

**10. Почему высокая Accuracy не всегда означает хорошую модель?**

Потому что при дисбалансе классов можно всегда предсказывать большинство и получать высокую Accuracy, не находя ни одного редкого случая. Accuracy смотрит на все ответы целиком, поэтому редкий, но важный класс может быть полностью пропущен, а метрика этого не покажет.